In [2]:
!pip install SpeechRecognition pyttsx3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 54.4 MB/s eta 0:00:00


In [3]:
import speech_recognition as sr
import pyttsx3
import webbrowser
from datetime import datetime, timedelta

In [6]:
!apt-get update -qq
!apt-get install -y espeak-ng

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 24 not upgraded.
Need to get 5,128 kB of archives.
After this operation, 13.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libpcaudio0 amd64 1.2-2build3 [9,144 B]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libsonic0 amd64 0.2.0-13build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 espeak-ng-data amd64 1.51+dfsg-12build1 [4,538 kB]
Get:4 http://archive.ubuntu.com/

In [7]:
import pyttsx3

engine = pyttsx3.init()
engine.say("Hello! Your voice assistant is working.")
engine.runAndWait()

In [8]:
!pip install -q gTTS

from gtts import gTTS
from IPython.display import Audio, display

tts = gTTS("Hello! Your voice assistant is working.", lang="en")
tts.save("/content/assistant_voice.mp3")

display(Audio("/content/assistant_voice.mp3", autoplay=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [18]:
!pip install SpeechRecognition pyttsx3 pyaudio
!python project4_voice_assistant.py

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)
python3: can't open file '/content/project4_voice_assistant.py': [Errno 2] No such file or directory


In [19]:
import speech_recognition as sr
import webbrowser
from datetime import datetime, timedelta
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
import base64

# Helper javascript code for recording microphone in browser
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async () => {
    blob = new Blob(chunks);
    text = await b2text(blob);
    resolve(text);
  };
  recorder.stop();
});
"""

def record_audio_browser(duration_seconds=5):
    print("Listening...")
    display(Javascript(RECORD_JS))
    # Execute js function and get base64 encoded string
    b64_string = output.eval_js(f'record({duration_seconds * 1000})')
    # Decode the base64 string to audio bytes
    audio_bytes = base64.b64decode(b64_string.split(',')[1])

    with open('/content/temp_input.wav', 'wb') as f:
        f.write(audio_bytes)
    return '/content/temp_input.wav'

reminders = []

def speak(message):
    print("Assistant:", message)
    # Generate Audio and play it in the notebook
    tts = gTTS(text=message, lang='en')
    tts.save("/content/response.mp3")
    display(Audio("/content/response.mp3", autoplay=True))

def listen():
    try:
        # Record audio from web browser and save to a temporary file
        audio_file_path = record_audio_browser(5)

        recognizer = sr.Recognizer()
        with sr.AudioFile(audio_file_path) as source:
            audio_data = recognizer.record(source)

        command = recognizer.recognize_google(audio_data).lower()
        print(f"You said: {command}")
        return command
    except Exception as e:
        speak("Sorry, I didn't catch that or couldn't process the audio.")
        return ""

speak("Hello! I am your personal assistant. Click run again to send commands.")

# We will do a single-step loop option for Colab to prevent endless blocking execution
command = listen()
if command:
    if "hello" in command:
        speak("Hello! How can I help you?")
    elif "open youtube" in command:
        speak("Opening YouTube.")
    elif "remind me to" in command:
        task = command.replace("remind me to ", "", 1)
        reminders.append({
            "task": task,
            "time": datetime.now() + timedelta(minutes=1)
        })
        speak(f"Reminder set for: {task}")
    else:
        speak("I do not know that command yet.")


Assistant: Hello! I am your personal assistant. Click run again to send commands.


Listening...


<IPython.core.display.Javascript object>

Assistant: Sorry, I didn't catch that or couldn't process the audio.


In [20]:
!pip install SpeechRecognition pyttsx3 pyaudio
!python project4_voice_assistant.py

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)
python3: can't open file '/content/project4_voice_assistant.py': [Errno 2] No such file or directory


In [21]:
%%writefile /content/project4_voice_assistant.py

import webbrowser
from datetime import datetime
import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display
import os

def speak(text):
    print("Assistant:", text)
    tts = gTTS(text=text, lang="en")
    tts.save("/content/assistant_voice.mp3")
    display(Audio("/content/assistant_voice.mp3", autoplay=False))

def listen():
    recognizer = sr.Recognizer()

    print("Enter your command below:")
    command = input("You: ").lower().strip()
    return command

def run_assistant():
    speak("Hello! Your personal assistant is ready.")

    while True:
        command = listen()

        if "time" in command:
            current_time = datetime.now().strftime("%I:%M %p")
            speak(f"The current time is {current_time}")

        elif "date" in command:
            today = datetime.now().strftime("%d %B %Y")
            speak(f"Today's date is {today}")

        elif "google" in command:
            webbrowser.open("https://www.google.com")
            speak("Opening Google")

        elif command in ["exit", "quit", "stop"]:
            speak("Goodbye!")
            break

        else:
            speak("Sorry, I do not understand that command yet.")

run_assistant()

Writing /content/project4_voice_assistant.py


In [22]:
!pip install -q SpeechRecognition gTTS

In [23]:
!ls -l /content/project4_voice_assistant.py

-rw-r--r-- 1 root root 1233 Oct 11 06:19 /content/project4_voice_assistant.py


In [24]:
!pip install -q SpeechRecognition gTTS

In [25]:
%run /content/project4_voice_assistant.py

Assistant: Hello! Your personal assistant is ready.


Enter your command below:
You: Hello
Assistant: Sorry, I do not understand that command yet.


Enter your command below:
You: can you hear me
Assistant: Sorry, I do not understand that command yet.


Enter your command below:
You: time
Assistant: The current time is 06:24 AM


Enter your command below:
You: date
Assistant: Today's date is 11 October 2026


Enter your command below:
You: google
Assistant: Opening Google


Enter your command below:
You: exit
Assistant: Goodbye!
